# 02 — Offline generation (Kaggle GPU, Internet OFF)

Notebook **standalone**: không cần repo hay code dataset. Add Input gồm frozen context dataset và đúng private model dataset, rồi đổi ba biến config bên dưới.


In [ ]:
# ===== CONFIG DUY NHẤT CẦN ĐỔI =====
RUNNER = 'B'               # B | C | D
MODEL_KEY = 'qwen25_7b'    # qwen25_7b | gemma3_4b
RUN_MODE = 'smoke'         # smoke | official

RUNNER_CONFIGS = {
    'B': 'report_shortlist_3::graph_dense_rrf',
    'C': 'report_shortlist_3::semantic_gs_rrf_rerank_k40',
    'D': 'report_shortlist_3::semantic_gs_rrf_no_rerank_reference',
}
MODEL_REGISTRY = {
    'qwen25_7b': 'Qwen/Qwen2.5-7B-Instruct',
    'gemma3_4b': 'google/gemma-3-4b-it',
}

# Chỉ điền khi auto-detect báo nhiều hơn một input phù hợp.
BUNDLE_INPUT = None       # folder chứa bundle_manifest.json hoặc file context_bundle_v1.zip
MODEL_ASSET_DIR = None    # folder chứa asset_manifest.json

assert RUNNER in RUNNER_CONFIGS
assert MODEL_KEY in MODEL_REGISTRY
assert RUN_MODE in {'smoke', 'official'}


In [ ]:
from pathlib import Path
import json, zipfile

def safe_extract(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    matches = sorted({p.parent for p in destination.rglob('bundle_manifest.json')})
    assert len(matches) == 1, f'ZIP phải chứa đúng một bundle: {matches}'
    return matches[0]

def resolve_bundle(explicit=None):
    if explicit:
        path = Path(explicit)
        if path.is_file() and path.suffix.lower() == '.zip':
            return safe_extract(path, '/kaggle/working/_context_bundle_input')
        assert (path / 'bundle_manifest.json').exists(), path
        return path
    raw = sorted({p.parent for p in Path('/kaggle/input').rglob('bundle_manifest.json')})
    if len(raw) == 1:
        return raw[0]
    archives = []
    for path in Path('/kaggle/input').rglob('*.zip'):
        try:
            with zipfile.ZipFile(path) as archive:
                if any(Path(name).name == 'bundle_manifest.json' for name in archive.namelist()):
                    archives.append(path)
        except zipfile.BadZipFile:
            pass
    assert len(archives) == 1, f'Hãy đặt BUNDLE_INPUT; raw={raw}, zip={archives}'
    return safe_extract(archives[0], '/kaggle/working/_context_bundle_input')

def resolve_model_asset(expected_model_id, explicit=None):
    candidates = [Path(explicit)] if explicit else sorted({p.parent for p in Path('/kaggle/input').rglob('asset_manifest.json')})
    matches = []
    for path in candidates:
        manifest_path = path / 'asset_manifest.json'
        if manifest_path.exists() and json.loads(manifest_path.read_text(encoding='utf-8')).get('model_id') == expected_model_id:
            matches.append(path)
    assert len(matches) == 1, f'Hãy đặt MODEL_ASSET_DIR cho {expected_model_id}; tìm thấy {matches}'
    return matches[0]

BUNDLE_DIR = resolve_bundle(BUNDLE_INPUT)
MODEL_ASSET_DIR = resolve_model_asset(MODEL_REGISTRY[MODEL_KEY], MODEL_ASSET_DIR)
print({'bundle': str(BUNDLE_DIR), 'model_assets': str(MODEL_ASSET_DIR)})


In [ ]:
# AUTO-EMBEDDED: standalone runtime; no repo import is required.

import hashlib
import json
import os
import platform
import tempfile
from pathlib import Path
from typing import Any, Iterable, Iterator


KIT_SCHEMA_VERSION = "local-llm-ablation-kit-v2"
BUNDLE_SCHEMA_VERSION = "model-agnostic-context-bundle-v2"
PREDICTION_SCHEMA_VERSION = "local-llm-predictions-v2"
JUDGE_SCHEMA_VERSION = "local-llm-gemini-judge-v2"


def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=json_default)


def json_default(value: Any) -> Any:
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def sha256_file(path: Path, *, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def stable_pair_id(*parts: str) -> str:
    raw = "::".join(str(part) for part in parts)
    return sha256_text(raw)[:24]


def shard_for_pair(pair_id: str, num_shards: int) -> int:
    if num_shards < 1:
        raise ValueError("num_shards must be positive")
    return int(sha256_text(pair_id)[:16], 16) % num_shards


def iter_json_records(path: Path) -> Iterator[dict[str, Any]]:
    """Read JSONL, pretty multi-line concatenated JSON, or a JSON array."""
    content = path.read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    offset = 0
    while offset < len(content):
        while offset < len(content) and (content[offset].isspace() or content[offset] == ","):
            offset += 1
        if offset >= len(content):
            break
        if content[offset] == "[":
            payload = json.loads(content)
            if not isinstance(payload, list):
                raise ValueError(f"Expected JSON array in {path}")
            for record in payload:
                if isinstance(record, dict):
                    yield record
            return
        payload, next_offset = decoder.raw_decode(content, offset)
        if not isinstance(payload, dict):
            raise ValueError(f"Expected JSON object at character {offset} in {path}")
        yield payload
        offset = next_offset


def load_jsonl_map(path: Path, key: str) -> dict[str, dict[str, Any]]:
    result: dict[str, dict[str, Any]] = {}
    if not path.exists():
        return result
    for record in iter_json_records(path):
        value = str(record.get(key) or "")
        if not value:
            raise ValueError(f"Record in {path} is missing key {key}")
        if value in result and canonical_json(result[value]) != canonical_json(record):
            raise ValueError(f"Conflicting duplicate {key}={value} in {path}")
        result[value] = record
    return result


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", newline="\n") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True, default=json_default))
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())


def atomic_write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="\n") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True, default=json_default)
            handle.write("\n")
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def write_jsonl_atomic(path: Path, records: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="\n") as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True, default=json_default))
                handle.write("\n")
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def resolve_directory(
    explicit: str | Path | None,
    *,
    marker: str,
    search_roots: Iterable[str | Path] = ("/kaggle/input", "."),
) -> Path:
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not (candidate / marker).exists():
            raise FileNotFoundError(f"{candidate} does not contain {marker}")
        return candidate
    matches: list[Path] = []
    for root in search_roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        matches.extend(path.parent.resolve() for path in root_path.rglob(marker))
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected exactly one directory containing {marker}; found {unique}")
    return unique[0]


def environment_summary() -> dict[str, Any]:
    summary: dict[str, Any] = {
        "python": platform.python_version(),
        "platform": platform.platform(),
    }
    for package in ("torch", "transformers", "accelerate", "bitsandbytes", "safetensors"):
        try:
            module = __import__(package)
            summary[package] = str(getattr(module, "__version__", "unknown"))
        except Exception:
            summary[package] = None
    try:
        import torch

        summary["cuda_available"] = torch.cuda.is_available()
        summary["cuda_device_count"] = torch.cuda.device_count()
        summary["cuda_devices"] = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
    except Exception:
        summary["cuda_available"] = False
        summary["cuda_device_count"] = 0
        summary["cuda_devices"] = []
    return summary


def assert_no_secret_keys(payload: Any) -> None:
    forbidden = {"api_key", "gemini_api_key", "hf_token", "token", "password", "neo4j_password"}

    def walk(value: Any, path: str) -> None:
        if isinstance(value, dict):
            for key, child in value.items():
                normalized = str(key).strip().lower()
                if normalized in forbidden and child not in (None, "", False):
                    raise ValueError(f"Secret-like value found at {path}.{key}; never persist secrets in artifacts")
                walk(child, f"{path}.{key}")
        elif isinstance(value, list):
            for index, child in enumerate(value):
                walk(child, f"{path}[{index}]")

    walk(payload, "root")


import json
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any



def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def safe_slug(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return slug or "model"


def _install_wheelhouse(asset_root: Path) -> None:
    wheelhouse = asset_root / "wheelhouse"
    wheels = sorted(wheelhouse.glob("*.whl")) if wheelhouse.exists() else []
    if not wheels:
        return
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *map(str, wheels)],
        check=True,
    )


def _load_latest_records(path: Path) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    if not path.exists():
        return latest
    for record in iter_json_records(path):
        pair_id = str(record.get("pair_id") or "")
        if not pair_id:
            continue
        previous = latest.get(pair_id)
        if previous is None or previous.get("status") != "completed" or record.get("status") == "completed":
            latest[pair_id] = record
    return latest


def _quantization_kwargs(
    quantization: str,
    torch: Any,
    BitsAndBytesConfig: Any,
    *,
    compute_dtype: Any,
) -> dict[str, Any]:
    if quantization == "4bit":
        return {
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=compute_dtype,
            )
        }
    if quantization in {"fp16", "float16"}:
        return {"torch_dtype": torch.float16}
    raise ValueError("quantization must be '4bit' or 'fp16'")


def _load_runtime(model_dir: Path, loader: str, model_kwargs: dict[str, Any]) -> tuple[Any, Any, Any]:
    import transformers

    if loader == "causal_lm":
        tokenizer = transformers.AutoTokenizer.from_pretrained(
            model_dir, local_files_only=True, trust_remote_code=False
        )
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id
        model = transformers.AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs)

        def encode(prompt: str) -> tuple[dict[str, Any], Any]:
            rendered = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            return tokenizer(rendered, return_tensors="pt", add_special_tokens=False), tokenizer

        return model, tokenizer, encode

    if loader == "gemma3_conditional":
        processor = transformers.AutoProcessor.from_pretrained(
            model_dir, local_files_only=True, trust_remote_code=False
        )
        model_class = getattr(transformers, "Gemma3ForConditionalGeneration", None)
        if model_class is None:
            model_class = getattr(transformers, "AutoModelForImageTextToText", None)
        if model_class is None:
            raise RuntimeError("Gemma 3 requires transformers with Gemma3ForConditionalGeneration support")
        model = model_class.from_pretrained(model_dir, **model_kwargs)
        tokenizer = processor.tokenizer
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id

        def encode(prompt: str) -> tuple[dict[str, Any], Any]:
            messages = [
                {
                    "role": "user",
                    "content": [{"type": "text", "text": prompt}],
                }
            ]
            inputs = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
            return dict(inputs), tokenizer

        return model, tokenizer, encode

    raise ValueError(f"Unsupported loader={loader!r}")


def run_offline_inference(config: dict[str, Any]) -> dict[str, Any]:
    """Generate deterministic answers for a frozen, model-agnostic retrieval bundle."""
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    bundle_dir = resolve_directory(config.get("bundle_dir"), marker="bundle_manifest.json")
    asset_root = resolve_directory(config.get("model_asset_dir"), marker="asset_manifest.json")
    bundle_manifest = json.loads((bundle_dir / "bundle_manifest.json").read_text(encoding="utf-8"))
    asset_manifest = json.loads((asset_root / "asset_manifest.json").read_text(encoding="utf-8"))
    if bundle_manifest["files"]["cases"]["sha256"] != sha256_file(bundle_dir / "cases.jsonl"):
        raise RuntimeError("cases.jsonl checksum does not match bundle_manifest.json")
    expected_model_id = str(config.get("expected_model_id") or "").strip()
    if expected_model_id and asset_manifest.get("model_id") != expected_model_id:
        raise RuntimeError(
            f"Mounted model is {asset_manifest.get('model_id')!r}, expected {expected_model_id!r}"
        )
    model_dir = asset_root / str(asset_manifest.get("model_subdir") or "model")
    if bool(config.get("install_wheelhouse", False)):
        _install_wheelhouse(asset_root)

    import torch
    from transformers import BitsAndBytesConfig, set_seed

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for the official inference run")

    shard_id = int(config.get("shard_id", 0))
    num_shards = int(config.get("num_shards", 1))
    if shard_id < 0 or shard_id >= num_shards:
        raise ValueError(f"shard_id must be in [0, {num_shards - 1}]")
    selected_suites = {str(value) for value in config.get("suites") or []}
    selected_config_keys = {str(value) for value in config.get("selected_config_keys") or []}
    limit = config.get("limit")
    seed = int(config.get("seed", 42))
    max_new_tokens = int(config.get("max_new_tokens", 1024))
    max_input_tokens = int(config.get("max_input_tokens", 24576))
    quantization = str(config.get("quantization") or "4bit").lower()
    retry_errors = bool(config.get("retry_errors", True))
    model_id = str(asset_manifest["model_id"])
    model_key = str(asset_manifest.get("model_key") or safe_slug(model_id))
    loader = str(asset_manifest.get("loader") or "causal_lm")
    # Gemma 3 is released in BF16 and can overflow/produce NaNs with FP16.
    # Qwen keeps the FP16 path used by the already validated run.
    compute_dtype = torch.bfloat16 if loader == "gemma3_conditional" else torch.float16
    compute_dtype_name = "bfloat16" if compute_dtype == torch.bfloat16 else "float16"
    selection_slug = (
        safe_slug(next(iter(selected_config_keys)).split("::")[-1])
        if len(selected_config_keys) == 1
        else "all-configs"
    )
    output_root = Path(config.get("output_root") or "/kaggle/working/local_llm_ablation").resolve()
    output_dir = output_root / model_key / selection_slug / f"shard_{shard_id:02d}_of_{num_shards:02d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions_path = output_dir / f"predictions_shard_{shard_id:02d}.jsonl"

    cases = [record for record in iter_json_records(bundle_dir / "cases.jsonl") if record.get("status") == "completed"]
    if selected_suites:
        cases = [record for record in cases if str(record.get("suite")) in selected_suites]
    if selected_config_keys:
        cases = [record for record in cases if str(record.get("config_key")) in selected_config_keys]
    assigned = [record for record in cases if shard_for_pair(str(record["pair_id"]), num_shards) == shard_id]
    assigned.sort(key=lambda record: str(record["pair_id"]))
    if limit is not None:
        assigned = assigned[: int(limit)]
    if not assigned:
        raise RuntimeError("No cases assigned; check selected_config_keys, shard_id and num_shards")

    latest = _load_latest_records(predictions_path)
    pending = [
        record
        for record in assigned
        if record["pair_id"] not in latest
        or (latest[record["pair_id"]].get("status") != "completed" and retry_errors)
    ]

    set_seed(seed)
    model_kwargs: dict[str, Any] = {
        "device_map": "auto",
        "local_files_only": True,
        "trust_remote_code": False,
        "low_cpu_mem_usage": True,
        **_quantization_kwargs(
            quantization,
            torch,
            BitsAndBytesConfig,
            compute_dtype=compute_dtype,
        ),
    }
    if loader == "gemma3_conditional":
        # Keep Gemma's non-quantized vision/projector modules in BF16 too.
        model_kwargs["torch_dtype"] = compute_dtype
    model, tokenizer, encode = _load_runtime(model_dir, loader, model_kwargs)
    model.eval()

    started_at = utc_now()
    for index, case in enumerate(pending, start=1):
        pair_id = str(case["pair_id"])
        pair_started = time.perf_counter()
        try:
            inputs, decode_tokenizer = encode(str(case["prompt"]))
            input_tokens = int(inputs["input_ids"].shape[-1])
            if input_tokens > max_input_tokens:
                raise ValueError(
                    f"Input has {input_tokens} tokens, exceeding max_input_tokens={max_input_tokens}; "
                    "evaluation prompts are never silently truncated"
                )
            model_device = next(model.parameters()).device
            inputs = {key: value.to(model_device) if hasattr(value, "to") else value for key, value in inputs.items()}
            torch.cuda.synchronize()
            generation_started = time.perf_counter()
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=max_new_tokens,
                    pad_token_id=decode_tokenizer.pad_token_id,
                    use_cache=True,
                )
            torch.cuda.synchronize()
            generation_ms = round((time.perf_counter() - generation_started) * 1000, 2)
            continuation = generated[0, input_tokens:]
            answer = decode_tokenizer.decode(continuation, skip_special_tokens=True).strip()
            if not answer:
                token_ids = continuation.detach().cpu().tolist()
                raw_decode = decode_tokenizer.decode(continuation, skip_special_tokens=False)
                raise RuntimeError(
                    "Model returned an empty answer; "
                    f"compute_dtype={compute_dtype_name}; input_tokens={input_tokens}; "
                    f"generated_shape={list(generated.shape)}; "
                    f"continuation_token_ids={token_ids[:32]!r}; raw_decode={raw_decode[:200]!r}"
                )
            prediction_id = stable_pair_id(
                pair_id,
                model_id,
                str(asset_manifest["resolved_revision"]),
                quantization,
                str(seed),
            )
            record = {
                "schema_version": PREDICTION_SCHEMA_VERSION,
                "status": "completed",
                "prediction_id": prediction_id,
                "pair_id": pair_id,
                "suite": case["suite"],
                "config_key": case["config_key"],
                "item_id": case["item_id"],
                "prompt_sha256": case["prompt_sha256"],
                "model_key": model_key,
                "model_id": model_id,
                "model_revision": asset_manifest["resolved_revision"],
                "loader": loader,
                "quantization": quantization,
                "compute_dtype": compute_dtype_name,
                "seed": seed,
                "do_sample": False,
                "input_tokens": input_tokens,
                "output_tokens": int(continuation.shape[-1]),
                "generation_latency_ms": generation_ms,
                "total_pair_latency_ms": round((time.perf_counter() - pair_started) * 1000, 2),
                "answer": answer,
                "completed_at": utc_now(),
            }
        except Exception as exc:
            record = {
                "schema_version": PREDICTION_SCHEMA_VERSION,
                "status": "failed",
                "pair_id": pair_id,
                "suite": case["suite"],
                "config_key": case["config_key"],
                "item_id": case["item_id"],
                "prompt_sha256": case["prompt_sha256"],
                "model_key": model_key,
                "model_id": model_id,
                "model_revision": asset_manifest["resolved_revision"],
                "loader": loader,
                "quantization": quantization,
                "compute_dtype": compute_dtype_name,
                "error_type": type(exc).__name__,
                "error": str(exc)[:2000],
                "failed_at": utc_now(),
            }
        append_jsonl(predictions_path, record)
        latest[pair_id] = record
        if index % 10 == 0 or index == len(pending):
            print(f"model={model_key} processed={index}/{len(pending)} pair_id={pair_id} status={record['status']}")

    compacted = [latest[record["pair_id"]] for record in assigned if record["pair_id"] in latest]
    write_jsonl_atomic(predictions_path, compacted)
    completed = sum(record.get("status") == "completed" for record in compacted)
    failed = sum(record.get("status") == "failed" for record in compacted)
    summary = {
        "schema_version": PREDICTION_SCHEMA_VERSION,
        "started_at": started_at,
        "completed_at": utc_now(),
        "shard_id": shard_id,
        "num_shards": num_shards,
        "selected_suites": sorted(selected_suites),
        "selected_config_keys": sorted(selected_config_keys),
        "assigned_pair_count": len(assigned),
        "completed_pair_count": completed,
        "failed_pair_count": failed,
        "is_complete": completed == len(assigned) and failed == 0,
        "model_key": model_key,
        "model_id": model_id,
        "model_revision": asset_manifest["resolved_revision"],
        "loader": loader,
        "quantization": quantization,
        "compute_dtype": compute_dtype_name,
        "seed": seed,
        "max_input_tokens": max_input_tokens,
        "max_new_tokens": max_new_tokens,
        "bundle_cases_sha256": bundle_manifest["files"]["cases"]["sha256"],
        "environment": environment_summary(),
        "predictions_file": predictions_path.name,
        "predictions_sha256": sha256_file(predictions_path),
    }
    atomic_write_json(output_dir / "shard_summary.json", summary)
    archive_base = output_root / (
        f"local_llm_predictions_{model_key}_{selection_slug}_{shard_id:02d}_of_{num_shards:02d}"
    )
    archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=output_dir))
    summary["archive_path"] = str(archive_path)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


In [ ]:
INFERENCE_CONFIG = {
    'bundle_dir': str(BUNDLE_DIR),
    'model_asset_dir': str(MODEL_ASSET_DIR),
    'expected_model_id': MODEL_REGISTRY[MODEL_KEY],
    'suites': ['report_shortlist_3'],
    'selected_config_keys': [RUNNER_CONFIGS[RUNNER]],
    'shard_id': 0,
    'num_shards': 1,
    'limit': 2 if RUN_MODE == 'smoke' else None,
    'seed': 42,
    'quantization': '4bit',
    'max_input_tokens': 24576,
    'max_new_tokens': 1024,
    'retry_errors': True,
    'install_wheelhouse': True,
    'output_root': '/kaggle/working/local_llm_ablation_smoke' if RUN_MODE == 'smoke' else '/kaggle/working/local_llm_ablation',
}
summary = run_offline_inference(INFERENCE_CONFIG)


In [ ]:
expected = 2 if RUN_MODE == 'smoke' else 100
assert summary['assigned_pair_count'] == expected, summary
assert summary['completed_pair_count'] == expected, summary
assert summary['failed_pair_count'] == 0, summary
assert summary['is_complete'], summary
print('PASS — tải ZIP này về gửi A:', summary['archive_path'])
